In [1]:
import matplotlib.pyplot as plt
import sunpy.map
import os
import re
from datetime import datetime
from matplotlib.patches import Wedge
from astropy.visualization import ImageNormalize, LinearStretch
from astropy.coordinates import SkyCoord
import astropy.units as u
import Sunlimb
import numpy as np
import image as img
from matplotlib.colors import Normalize, LogNorm
import sunkit_image.enhance as enhance
%matplotlib notebook

定义一些绘图函数

In [2]:
# 绘制EUI图像
def draw_fig(sun_map, colorbar=False, limb=False, grid=False, rotate=False, process=None, return_fig=False, **kwargs):
    if type(sun_map) == str:
        sun_map = sunpy.map.Map(sun_map)
    if rotate:
        sun_map = sun_map.rotate()
    if process is not None:
        map_data = process(sun_map.data)
        sun_map = sunpy.map.Map(map_data, sun_map.fits_header)
    fig = plt.figure()
    ax = fig.add_subplot(projection=sun_map)
    im = sun_map.plot(axes=ax, **kwargs)
    if colorbar:
        plt.colorbar(im, ax=ax)
    if limb:
        sun_map.draw_limb(axes=ax)
    if grid:
        sun_map.draw_grid(axes=ax)
    if return_fig:
        return fig

# unsharp mask处理
def unsharp_mask(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    unsharp_mask_image = img.unsharp_mask(image_nan_to_0, kernel_size=11)
    unsharp_mask_image[np.isnan(image)] = np.nan
    return unsharp_mask_image

# atrous小波处理
def atrous(image):
    image_nan_to_0 = np.float32(np.nan_to_num(image, nan=0.0))
    coe = img.a_trous(image_nan_to_0, level=4)
    hq = coe.data[1]+coe.data[2]
    hq = img.box_car_smoothing(hq, kernel_size=7)
    hq[np.isnan(image)] = np.nan
    return hq

def wow(image):
    return enhance.wow(image, denoise_coefficients=[5, 2, 1], soft_threshold=True, bilateral=1, gamma=1)

# 图像裁剪
def cut_map(hri, bottom_left, top_right):
    if type(hri) is str:
        hri = sunpy.map.Map(hri)
    bottom_left = SkyCoord(bottom_left[0]*u.arcsec, bottom_left[1]*u.arcsec, frame=hri.coordinate_frame)
    top_right = SkyCoord(top_right[0]*u.arcsec, top_right[1]*u.arcsec, frame=hri.coordinate_frame)
    
    submap = hri.submap(bottom_left, top_right=top_right)
    return submap

# 目的
希望利用HRI EUV仪器高分辨率的日冕边缘的数据制作时空切片，并利用小波变换等方法，研究日冕中波动的传播特征
# 数据
使用的数据是solar obiter的HRI EUV望远镜174波段于2022年3月30日4:45-4:55 UCT期间拍摄的数据。数据时间间隔为3s，空间分辨率为0.492arcsec

In [3]:
file_path = "E:/python/projects/alfven/data/hrieuvopn/alignment_data/10moving/"
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]
def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min

files = sorted(files, key=extract_datetime)
files = files[295:495]

# 问题一：较大的噪声，难以发现“Y型”喷流
## 对时空图的检查：
### 时空图的制作：
在每张图像中取距离太阳边缘20Mm处，角度范围为179~182度的像素，并在时间上进行堆叠，制作所有数据的时空切片。其中角度的定义为：将太阳盘面置于极坐标系，太阳中心为极点，太阳北极的角度定为0度，顺时针方向为角度增加的方向。

In [ ]:
deg_range = (179, 182)
d_deg = 0.01
d_time = 3
deg_vect = np.arange(deg_range[0], deg_range[1] + d_deg / 2, d_deg)
time_vect = np.arange(0, len(files) * d_time + 0.5 * d_time, d_time) / 60  # unit: minute
r = 20
fig_st, ax_st = plt.subplots(figsize=(6, 4))

st_data = Sunlimb.space_time_plot(ax=ax_st, files=files, deg_range=deg_range, r=r, d_time=3, d_deg=0.01, process=wow,
                                  cmap='sdoaia171',
                                  title='space_time_plot r=20Mm', xlabel='degree', ylabel='time(min)', norm=Normalize(),
                                  colorbar=True)

fig_st.savefig('./fig/raw_space_time_plot.png', dpi=400)

<IPython.core.display.Javascript object>

Processing files:   0%|          | 0/200 [00:00<?, ?it/s]

上面的时空图是由原始数据绘制的，在时空图中较难发现波动特征。现在查看时空切片数据的信噪比

In [ ]:
snr_space_time = 10* np.log10(np.mean(st_data)/np.std(st_data))
print(f'SNR of space time plot: {snr_space_time} dB')

## 对EUI图像的检查
裁剪部分图像进行检查。图像中白色扇形的边缘代表时空切片在每张图中所取的像素的位置

In [ ]:
top_right = (100, -2850)
bottom_left = (-150, -3100)

example_map = sunpy.map.Map(files[10])
example_cut = cut_map(example_map, bottom_left, top_right)
example_fig = draw_fig(example_cut, cmap='sdoaia171', norm=Normalize(), return_fig=True)
example_ax = example_fig.axes[0]

r = example_cut.fits_header['RSUN_ARC'] * (20 * 1e6 / example_cut.fits_header['RSUN_REF'] + 1) / 3600
center = (0, 0)
begin = -92
end = -89
wedge = Wedge(center, r, begin, end, color="white", transform=example_ax.get_transform('world'), fill=False, linestyle='--')
example_ax.add_patch(wedge)
example_ax.invert_xaxis()
example_ax.invert_yaxis()

example_fig.savefig('./fig/example_map.png', dpi=400)

In [ ]:
snr_EUI = 10 * np.log10(np.mean(example_cut.data)/np.std(example_cut.data))
print(f'SNR of EUI photo: {snr_EUI} dB')

上图中几乎不可见日冕中的喷流，信噪比为3分贝，极低。现在对图像进行atrous小波处理。将第0级小波系数视为噪声丢弃，使用第一级和第二级小波系数相加作为新的数据，再对数据进行平滑核大小为3的平滑处理。将colorbar范围设置为-20~20以提高喷流可见度

In [ ]:
top_right = (100, -2850)
bottom_left = (-150, -3100)

example_map = sunpy.map.Map(files[15])
example_cut = cut_map(example_map, bottom_left, top_right)
example_fig = draw_fig(example_cut, cmap='sdoaia171', norm=Normalize(vmin=-20, vmax=20), return_fig=True, process=atrous)
example_ax = example_fig.axes[0]

r = example_cut.fits_header['RSUN_ARC'] * (20 * 1e6 / example_cut.fits_header['RSUN_REF'] + 1) / 3600
center = (0, 0)
begin = -92
end = -89
wedge = Wedge(center, r, begin, end, color="white", transform=example_ax.get_transform('world'), fill=False, linestyle='--')
example_ax.add_patch(wedge)
example_ax.invert_xaxis()
example_ax.invert_yaxis()

example_fig.savefig('./fig/example_map.png', dpi=400)

In [ ]:
snr_EUI = 10 * np.log10(np.mean(atrous(example_cut.data))/np.std(atrous(example_cut.data)))
print(f'SNR of EUI photo: {snr_EUI} dB')

# 问题二：图像抖动
## 制作原始图像未裁剪及裁剪后的视频

In [ ]:
from tqdm.notebook import tqdm

def draw_frames(euv_files, fig_path, colorbar=False, limb=False, grid=False, process=None, cut=None, **kwargs):
    if not os.path.exists(fig_path):
        os.makedirs(fig_path)
    fig = plt.figure()
    for file in tqdm(euv_files, desc='Drawing frames', total=len(euv_files)):
        fig.clear()
        if cut is not None:
            euv_map = cut(file)
        else:
            euv_map = sunpy.map.Map(file)
        if process is not None:
            map_data = process(euv_map.data)
            euv_map = sunpy.map.Map(map_data, euv_map.meta)
        ax = fig.add_subplot(projection=euv_map)
        im = euv_map.plot(axes=ax, **kwargs)
        if colorbar:
            plt.colorbar(im, ax=ax)
        if limb:
            euv_map.draw_limb(axes=ax)
        if grid:
            euv_map.draw_grid(axes=ax)
            
        r = euv_map.fits_header['RSUN_ARC'] * (20 * 1e6 / euv_map.fits_header['RSUN_REF'] + 1) / 3600
        center = (0, 0)
        begin = -92
        end = -89
    
        wedge = Wedge(center, r, begin, end, color="white", transform=ax.get_transform('world'), fill=False, linestyle='--')
        ax.add_patch(wedge)
        
        filename = euv_map.fits_header['filename']
        name = os.path.splitext(filename)[0]
        fig.savefig(fig_path + name + '.png', dpi=400)
        
top_right = (100, -2850)
bottom_left = (-150, -3100)
def cut(euv_file):
    return cut_map(euv_file, bottom_left, top_right)

### 原始图像未裁剪

In [ ]:
raw_files = files[300:500]
raw_fig_path = './movie/fig_raw/'
raw_movie_name = './movie/movie_raw.mp4'

draw_frames(raw_files, raw_fig_path, limb=True, norm=Normalize(vmin=0, vmax=2500), cmap='sdoaia171')

img.images_to_video(raw_fig_path, raw_movie_name, fps=20)

### 原始图像裁剪后

In [ ]:
raw_fig_cut_path = './movie/fig_raw_cut/'
raw_cut_movie_name = './movie/movie_raw_cut.mp4'

draw_frames(raw_files, raw_fig_cut_path, limb=True, norm=Normalize(vmin=0, vmax=2500), cmap='sdoaia171', cut=cut)

img.images_to_video(raw_fig_cut_path, raw_cut_movie_name, fps=20)

### 矫正过后的图像未裁剪

In [ ]:
offset_fig_path = './movie/fig_offset/'
offset_movie_name = './movie/movie_offset.mp4'

draw_frames(files, offset_fig_path, limb=True, norm=Normalize(vmin=0, vmax=2500), cmap='sdoaia171')

img.images_to_video(offset_fig_path, offset_movie_name, fps=20)

### 矫正过后的图像未裁剪

In [ ]:
offset_cut_fig_path = './movie/fig_offset_cut/'
offset_cut_movie_name = './movie/movie_offset_cut.mp4'

draw_frames(files, offset_cut_fig_path, limb=True, norm=Normalize(vmin=0, vmax=2500), cmap='sdoaia171', cut=cut)

img.images_to_video(offset_cut_fig_path, offset_cut_movie_name, fps=20)

# 问题三：图像信号增强
## 滑动平均
### 滑动平均图像未裁剪

In [ ]:
average_fig_path = './movie/fig_average/'
average_movie_name = './movie/movie_average.mp4'

draw_frames(files, average_fig_path, limb=True, norm=Normalize(vmin=0, vmax=2500), cmap='sdoaia171')

img.images_to_video(average_fig_path, average_movie_name, fps=20)

### 滑动平均图像裁剪后

In [ ]:
average_fig_cut_path = './movie/fig_average_cut/'
average_cut_movie_name = './movie/movie_average_cut.mp4'

draw_frames(files, average_fig_cut_path, limb=True, norm=Normalize(vmin=0, vmax=2500), cmap='sdoaia171', cut=cut)

img.images_to_video(average_fig_cut_path, average_cut_movie_name, fps=20)

## unsharp mask
先选取合适colorbar范围

In [ ]:
draw_fig(files[0], cmap='sdoaia171', colorbar=True, norm=Normalize(vmin=0, vmax=100), process=unsharp_mask)

In [ ]:
draw_fig(cut(files[0]), cmap='sdoaia171', colorbar=True, norm=Normalize(vmin=-150, vmax=150), process=unsharp_mask)

### unsharp mask图像未裁剪

In [ ]:
mask_fig_path = './movie/fig_mask/'
mask_movie_name = './movie/movie_mask.mp4'

draw_frames(files, mask_fig_path, limb=True, norm=Normalize(vmin=0, vmax=100), cmap='sdoaia171', process=unsharp_mask)

### unsharp mask图像裁剪后

In [ ]:
mask_cut_fig_path = './movie/fig_mask_cut/'
mask_cut_movie_name = './movie/movie_mask_cut.mp4'

draw_frames(files, mask_cut_fig_path, limb=True, norm=Normalize(vmin=-150, vmax=150), cmap='sdoaia171', process=unsharp_mask, cut=cut)

## atrous
对图像进行atrous处理，先选取合适colorbar范围

In [ ]:
draw_fig(files[0], cmap='sdoaia171', colorbar=True, norm=Normalize(vmin=-20, vmax=20), process=atrous)

In [ ]:
draw_fig(cut(files[0]), cmap='sdoaia171', colorbar=True, norm=Normalize(vmin=-20, vmax=20), process=atrous, return_fig=True).savefig('./fig/atrous_cut.png', dpi=400)

### atrous未裁剪

In [ ]:
atrous_fig_path = './movie/fig_atrous/'
atrous_movie_name = './movie/movie_atrous.gif'

draw_frames(files, atrous_fig_path, limb=True, norm=Normalize(vmin=-20, vmax=20), cmap='sdoaia171', process=atrous)

In [ ]:
img.images_to_gif(atrous_fig_path, atrous_movie_name, fps=10, width=800)

### atrous裁剪后

In [ ]:
atrous_cut_fig_path = './movie/fig_atrous_cut/'
atrous_cut_movie_name = './movie/movie_atrous_cut.gif'

draw_frames(files, atrous_cut_fig_path, limb=True, norm=Normalize(vmin=-20, vmax=20), cmap='sdoaia171', process=atrous, cut=cut)

In [ ]:
img.images_to_gif(atrous_cut_fig_path, atrous_cut_movie_name, fps=10, width=960)

### 与时空图进行对比

In [ ]:
img.images_to_gif("./fig/st_averaged/", './movie/movie_atrous_st.gif', fps=10, width=1100)